# 조건부 분기 + 비동기 실행 + 체크 포인팅 구현 RAG 기반 chatbot 구현

## 환경 설정

In [9]:
# 필요한 모듈 / 클래스 / 함수 임포트
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langgraph.graph import add_messages, StateGraph, END
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

In [2]:
# .env file -> API Key 가져오기
file_path='.env'
output=load_dotenv(file_path, override=True)
print(output)

True


In [3]:
# 공통 LLM 생성
llm = ChatOpenAI(model="gpt-5-nano")

## RAG 시스템 기반 설정

### 데이터 로드 및 변환

In [4]:
# 파일 경로 설정
csv_path = "faq_manual.csv"

# CSV 파일 로드 및 변환 도구 생성 -> CSVLoader 클래스 호출
loader = CSVLoader(
    file_path=csv_path,
    encoding="utf-8",
    # 본문(page_content)에 포함할 컬럼 지정
    content_columns=["question", "answer"]
)

# load() 함수 호출 -> Document로 변환
docs = loader.load()

print(f"로드된 총 문서 수: {len(docs)}개")
print("-" * 60)
print(f"[첫 번째 Document 본문(page_content)]\n{docs[0].page_content}")
print("-" * 60)
print(f"[첫 번째 Document 메타데이터(metadata)]\n{docs[0].metadata}")

로드된 총 문서 수: 10개
------------------------------------------------------------
[첫 번째 Document 본문(page_content)]
question: 코드크래프터스 아카데미에 어떻게 가입하나요?
answer: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.
------------------------------------------------------------
[첫 번째 Document 메타데이터(metadata)]
{'source': 'faq_manual.csv', 'row': 0}


### 텍스트 임베딩 모델 생성

In [5]:
# 모델 이름 설정
model_name = "text-embedding-3-small"

# OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

### 백터 저장소 및 검색 도구 생성

In [6]:
# 벡터 저장소 저장 경로 설정
chroma_path = 'chroma_db_multi'

# 이미 생성된 벡터 저장소가 있다면 불러오고, 없다면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

기존 벡터 저장소를 성공적으로 불러왔습니다.


In [7]:
# 검색 도구 생성
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

## 에이전트별 전용 체인 생성

In [10]:
# [의도 분류 체인] 이전 대화 맥락과 최신 질문을 보고 매뉴얼(백터 저장소) 검색의 필요 여부 판별

# system 프롬프트 작성
classifier_system_prompt = """
당신은 온라인 교육 플랫폼 '코드크래프터스 아카데미'의 고객 문의 분류 전문가입니다.
이전 대화 기록(chat history)과 최신 사용자 질문을 함께 고려하여,
사내 고객 응대 매뉴얼(수강 신청, 회원가입, 환불 규정, 결제 수단, 강의 자료, 수료증 등) 조회가 필요한지 판단하세요.

- 플랫폼 이용, 강좌 정책, 환불, 결제, 수료증 등 회사 매뉴얼 확인이 필요한 문의(이전 매뉴얼 대화의 후속 질문 포함): 'RAG'
- 단순 인사말, 감사 표현, 일반 프로그래밍 상식/문법 질문 등 사내 매뉴얼이 필요 없는 대화: 'NO_RAG'

반드시 'RAG' 또는 'NO_RAG' 단어 하나만 정확하게 출력하세요. 다른 설명이나 공백은 일체 포함하지 마세요.
"""

# 전체 프롬프트 작성 -> ChatPromptTemplate 클래스 -> 전용 함수: from_messages
classifier_prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", classifier_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

print(classifier_prompt)

# chain 생성
classifier_chain = classifier_prompt | llm | StrOutputParser()

input_variables=['chat_history', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messag

In [11]:
# [질문 재구성 체인] 이전 대화 매락을 참조하여 모호한 질문을 검색에 적합한 명확한 질문으로 변환

# system 프롬프트 생성
rephrase_system_prompt = """
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?"),
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: 수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?)으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
"""

# 전체 프롬프트 작성
rephrase_prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", rephrase_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

print(rephrase_prompt)

# chain 생성
rephrase_chain = rephrase_prompt | llm | StrOutputParser()

input_variables=['chat_history', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messag

In [13]:
# [답변 생성 체인] RAG 기반 고객 응대 체인 및 일반 응대 체인 생성

## 1) RAG 기반 답변 체인

### 시스템 프롬프트 작성
rag_qa_system_prompt = """
당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다.
아래 제공된 [매뉴얼 내용]과 이전 대화 맥락을 바탕으로 고객의 질문에 정중하고 명확하게 답변해 주세요.
매뉴얼에 명시된 내용을 왜곡하지 말고 사실에 기반하여 친절하게 안내하세요.

[매뉴얼 내용]
{context}
"""

### 전체 프롬프트 작성
rag_qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rag_qa_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

print(rag_qa_prompt)

### chain 생성 -> LLM 답변 -> AIMessage -> return {"messages":[AIMessage]} -> StrOutputParser() 생략
rag_qa_chain = rag_qa_prompt | llm

print('-'*80)

## 2) 일반 대화에 대한 답변 체인

### 시스템 프롬프트 생성
general_qa_system_prompt = """
당신은 친절하고 명쾌한 답변을 하는 AI 비서입니다.
이전 대화 맥락을 기억하며 사용자의 질문이나 인사에 자연스럽고 유익하게 답변해 주세요.
"""

### 전체 프롬프트 생성
general_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", general_qa_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

print(general_qa_prompt)

### chain 생성: 상태(messages)에 AIMessage 형태로 바로 누적하기 위해 StrOutputParser() 생략
general_qa_chain = general_qa_prompt | llm

input_variables=['chat_history', 'context', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_

## 그래프 상태 정의

In [14]:
# pydantic 라이브러리 -> BaseModel 클래스 상속 -> 커스텀 자료형(AgentState) 생성
class AgentState(BaseModel):
    # 대화 내역 누적 항목
    messages: Annotated[list[BaseMessage], add_messages] = []
    # 매뉴얼 검색(RAG) 필요 여부 저장
    needs_rag: bool = False
    # 검색된 매뉴얼 내용 저장
    context: str = ""

## 그래프 노드 함수 정의

In [ ]:
# 1. 질문 의도 분류 노드 (비동기 처리 방식)
async def classifier_node(state: AgentState) -> dict:
    print("[Node 1: 질문 의도 분류] 실행 중...")
    print('-'*80)

    # 대화 히스토리 -> 사용자 최신 질문 추출 -> state.messages -> [... ,HumanMessage]
    latest_question = state.messages[-1].content

    # 대화 히스토리 -> 사용자 최신 질문 이전 대화 내역 추출
    chat_history = state.messages[:-1]

    # 의도 분류 체인 호출 -> 비동기 처리 방식 -> await, ainvoke() -> 결과값: RAG or NO_RAG
    decision = await classifier_chain.ainvoke({
        "chat_history": chat_history,
        "input": latest_question
    })

    # 단답형 응답 문자열 -> 블리언 변환
    decision1 = decision.strip().upper()
    # if decision1 == "RAG":
    #     needs_rag=True
    # else:
    #     needs_rag=False
    needs_rag = (decision1=="RAG")

    print(f"LLM 판단 결과: {decision} (RAG 검색 필요 여부: {needs_rag})")
    print('-' * 80)

    # 결과 생성(return) -> dict 자료형 -> 상태 업데이트
    return {"needs_rag": needs_rag}

In [ ]:
# 2. 검색 노드 (비동기 처리 방식)
async def retriever_node(state: AgentState) -> dict:
    print("[Node 2: 사내 매뉴얼 검색 (RAG)] 실행 중...")
    print('-'*80)

    # 상태 -> 대화 히스토리(messages) -> 사용자 최신 질문 및 이전 대화 내역 추출
    # messages: [HumanMessage] / [HumanMessage, AIMessage, HumanMessage]
    latest_question = state.messages[-1].content
    chat_history = state.messages[:-1]

    # 이전 대화 기록이 존재 -> 질문 재구성 체인 실행 -> 재구성된 질문 생성 -> 비동기 처리 방식 -> await, ainvoke()
    if chat_history:
        query = await rephrase_chain.ainvoke({
            "chat_history": chat_history,
            "input": latest_question
        })
        print(f"[결과] 재구성된 검색 쿼리: {query}")

    else:
        # 사용자 -> 첫 번째 질문
        query = latest_question

    # 재구성된 질문 -> 검색 도구 실행